# Lecture 03: Embeddings and PyTorch for Language Models

CS40008.01 · Baojian Zhou · Fudan University · September 23, 2026

**Question:** How do discrete tokens become trainable representations?

This notebook follows the slides in order. Exercises E01–E04 match the slide IDs; P01–P03 are optional practices for after class. Everything runs offline on a CPU in a few seconds. E01 and the first half of E02 use only the Python standard library; the rest uses PyTorch, which `uv sync` already installed. All corpora and vectors here are **toy data** made for this lecture.

| Exercise | Slide | What you compute |
| --- | --- | --- |
| E01 | PPMI from a co-occurrence table | PMI and PPMI for three word–context pairs |
| E02 | One skip-gram update by hand | Loss and gradients for one positive and one negative pair, then the same with autograd |
| E03 | Predict the tensor shapes | Embedding lookup, one-hot product, batching, output projection, cross-entropy |
| E04 | Count the parameters | Embedding and output tables with and without weight sharing; memory in fp32 and bf16 |
| P01 | optional | Train skip-gram with negative sampling on a toy corpus |
| P02 | optional | Nearest neighbours and analogies on the toy vectors |
| P03 | optional | The five-line training loop on one batch |

## E01 · PPMI from a co-occurrence table

The counts are the word–context table from Jurafsky and Martin, *Speech and Language Processing* (3rd ed. draft of August 19, 2026), [Appendix J](https://web.stanford.edu/~jurafsky/slp3/J.pdf), Figure J.2, computed from Wikipedia; the same four words appear in Figure 5.3 of the embeddings chapter. Rows are words, columns are context words.

$$\mathrm{PMI}(w,c)=\log_2\frac{p(w,c)}{p(w)\,p(c)},\qquad \mathrm{PPMI}(w,c)=\max(\mathrm{PMI}(w,c),0)$$

Compute $\mathrm{PPMI}(\text{information},\text{data})$ by hand from the row, column, and grand totals, then run the cell.

In [ ]:
import math

CONTEXTS = ["computer", "data", "result", "pie", "sugar"]
COUNTS = {
    "cherry":      [2, 8, 9, 442, 25],
    "strawberry":  [0, 0, 1, 60, 19],
    "digital":     [1670, 1683, 85, 5, 4],
    "information": [3325, 3982, 378, 5, 13],
}

TOTAL = sum(sum(row) for row in COUNTS.values())
ROW_TOTAL = {word: sum(row) for word, row in COUNTS.items()}
COLUMN_TOTAL = {c: sum(COUNTS[w][j] for w in COUNTS) for j, c in enumerate(CONTEXTS)}


def pmi(word, context):
    # Pointwise mutual information in bits; -inf when the pair never co-occurs.
    joint = COUNTS[word][CONTEXTS.index(context)] / TOTAL
    if joint == 0:
        return float("-inf")
    return math.log2(joint / ((ROW_TOTAL[word] / TOTAL) * (COLUMN_TOTAL[context] / TOTAL)))


def ppmi(word, context):
    return max(pmi(word, context), 0.0)


print("grand total:", TOTAL, "| row totals:", ROW_TOTAL, "| column totals:", COLUMN_TOTAL)
for word, context in [("information", "data"), ("cherry", "pie"), ("strawberry", "computer"), ("digital", "pie")]:
    print(f"PMI({word}, {context}) = {pmi(word, context):8.4f}   PPMI = {ppmi(word, context):.4f}")

**Check:** $p(\text{information},\text{data}) = 3982/11716 = 0.3399$, $p(\text{information}) = 7703/11716 = 0.6575$, $p(\text{data}) = 5673/11716 = 0.4842$, so $\mathrm{PMI} = \log_2(0.3399/(0.6575 \times 0.4842)) = 0.0944$ bits. The pair is frequent, yet barely more frequent than chance, because both words are frequent. `cherry` and `pie` are rarer but strongly associated ($4.38$ bits). A zero count gives $-\infty$, and a negative PMI needs a large corpus to be reliable: PPMI sets both to $0$.

## E02 · One skip-gram update by hand

Skip-gram with negative sampling scores a (word, context) pair with a dot product and turns the score into a probability with the sigmoid. For a center word vector $\mathbf{w}$, one positive context $\mathbf{c}_{pos}$ and one sampled negative context $\mathbf{c}_{neg}$ (so $k=1$):

$$\ell = -\log\sigma(\mathbf{w}^\top\mathbf{c}_{pos}) - \log\sigma(-\mathbf{w}^\top\mathbf{c}_{neg})$$

$$\frac{\partial \ell}{\partial \mathbf{c}_{pos}} = (s_{pos}-1)\,\mathbf{w},\qquad
\frac{\partial \ell}{\partial \mathbf{c}_{neg}} = s_{neg}\,\mathbf{w},\qquad
\frac{\partial \ell}{\partial \mathbf{w}} = (s_{pos}-1)\,\mathbf{c}_{pos} + s_{neg}\,\mathbf{c}_{neg}$$

with $s_{pos}=\sigma(\mathbf{w}^\top\mathbf{c}_{pos})$ and $s_{neg}=\sigma(\mathbf{w}^\top\mathbf{c}_{neg})$. Use $\mathbf{w}=(1, 0.5)$, $\mathbf{c}_{pos}=(0.5, 1)$, $\mathbf{c}_{neg}=(1, -1)$ and learning rate $\eta = 0.5$. Compute the two scores and the loss by hand first.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + math.exp(-z))


def dot(a, b):
    return sum(x * y for x, y in zip(a, b))


W_VEC, C_POS, C_NEG, ETA = [1.0, 0.5], [0.5, 1.0], [1.0, -1.0], 0.5

s_pos = sigmoid(dot(W_VEC, C_POS))
s_neg = sigmoid(dot(W_VEC, C_NEG))
e02_loss = -math.log(s_pos) - math.log(1.0 - s_neg)  # sigma(-z) = 1 - sigma(z)

e02_grads = {
    "c_pos": [(s_pos - 1.0) * x for x in W_VEC],
    "c_neg": [s_neg * x for x in W_VEC],
    "w": [(s_pos - 1.0) * p + s_neg * n for p, n in zip(C_POS, C_NEG)],
}
e02_updated = {
    "c_pos": [v - ETA * g for v, g in zip(C_POS, e02_grads["c_pos"])],
    "c_neg": [v - ETA * g for v, g in zip(C_NEG, e02_grads["c_neg"])],
    "w": [v - ETA * g for v, g in zip(W_VEC, e02_grads["w"])],
}

print(f"scores: w.c_pos = {dot(W_VEC, C_POS):.1f}, w.c_neg = {dot(W_VEC, C_NEG):.1f}")
print(f"s_pos = {s_pos:.4f}, s_neg = {s_neg:.4f}, loss = {e02_loss:.4f}")
for name in ["c_pos", "c_neg", "w"]:
    grad = ", ".join(f"{g:+.4f}" for g in e02_grads[name])
    new = ", ".join(f"{v:+.4f}" for v in e02_updated[name])
    print(f"grad {name:5s} = ({grad})   updated {name:5s} = ({new})")

after = -math.log(sigmoid(dot(e02_updated["w"], e02_updated["c_pos"]))) - math.log(1.0 - sigmoid(dot(e02_updated["w"], e02_updated["c_neg"])))
print(f"loss after one step = {after:.4f}")

**Check:** $s_{pos}=\sigma(1)=0.7311$, $s_{neg}=\sigma(0.5)=0.6225$, $\ell = -\log 0.7311 - \log(1-0.6225) = 1.2873$. The update pulls $\mathbf{c}_{pos}$ toward $\mathbf{w}$ and pushes $\mathbf{c}_{neg}$ away from it; the loss falls after one step.

You will rarely derive such gradients by hand again. PyTorch records the operations of the forward pass and applies the chain rule for you (**autograd**). The next cell computes the same three gradients with `loss.backward()`.

In [ ]:
import torch
import torch.nn.functional as F

w = torch.tensor(W_VEC, requires_grad=True)
c_pos = torch.tensor(C_POS, requires_grad=True)
c_neg = torch.tensor(C_NEG, requires_grad=True)

loss = -F.logsigmoid(w @ c_pos) - F.logsigmoid(-(w @ c_neg))
loss.backward()

e02_autograd = {"c_pos": c_pos.grad.tolist(), "c_neg": c_neg.grad.tolist(), "w": w.grad.tolist()}
print(f"loss = {loss.item():.4f}")
for name, grad in e02_autograd.items():
    print(f"autograd {name:5s} =", [round(g, 4) for g in grad], "| by hand =", [round(g, 4) for g in e02_grads[name]])

## E03 · Predict the tensor shapes

A language model never sees strings. It sees a batch of token IDs with shape $(B, T)$: $B$ sequences of $T$ tokens. The embedding table $E\in\mathbb{R}^{|V|\times d}$ turns each ID into a row of $E$.

Before running each cell, **write down the shape you expect**. Here $|V|=10$, $d=4$, $B=2$, $T=3$.

In [ ]:
torch.manual_seed(0)
V, D, B, T = 10, 4, 2, 3

embedding = torch.nn.Embedding(V, D)
token_ids = torch.tensor([[1, 5, 5], [7, 1, 0]])      # (B, T), integer IDs

x = embedding(token_ids)
print("E.weight :", tuple(embedding.weight.shape))
print("token_ids:", tuple(token_ids.shape), token_ids.dtype)
print("E[ids]   :", tuple(x.shape))

# A lookup is a one-hot vector times E, without ever building the one-hot vector.
one_hot = F.one_hot(token_ids, num_classes=V).float()  # (B, T, |V|)
same = torch.allclose(one_hot @ embedding.weight, x)
print("one_hot  :", tuple(one_hot.shape), "| one_hot @ E equals E[ids]:", same)
e03_shapes = {"weight": tuple(embedding.weight.shape), "x": tuple(x.shape), "one_hot": tuple(one_hot.shape)}
e03_lookup_equals_onehot = same

**Output projection.** To predict the next token the model needs one score (a *logit*) per vocabulary entry at every position. A second table $W_{out}\in\mathbb{R}^{|V|\times d}$ maps each $d$-dimensional vector back to $|V|$ scores: `logits = x @ W_out.T`. Softmax turns the scores into $p_\theta(\cdot\mid\text{context})$, and the training loss is Lecture 02's negative log-likelihood, averaged over the $B\times T$ positions.

Predict: the shape of `logits`, and the loss when $W_{out}$ is all zeros.

In [ ]:
w_out = torch.zeros(V, D, requires_grad=True)          # (|V|, d), zeros on purpose
targets = torch.tensor([[5, 5, 2], [1, 0, 3]])         # (B, T), the next token at each position

logits = x @ w_out.T                                   # (B, T, d) @ (d, |V|) -> (B, T, |V|)
loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))

log_probs = torch.log_softmax(logits, dim=-1)
by_hand = -log_probs.gather(-1, targets.unsqueeze(-1)).mean()

print("logits:", tuple(logits.shape))
print(f"cross-entropy = {loss.item():.4f} | by hand = {by_hand.item():.4f} | log|V| = {math.log(V):.4f}")
e03_logits_shape, e03_uniform_loss = tuple(logits.shape), loss.item()

**Check:** zero logits give the uniform distribution, so the loss is $\log 10 = 2.3026$ nats and the perplexity is $10$: the digits example of Lecture 02. A freshly initialized language model should start near $\log|V|$; a much larger first loss signals a bug.

**Which rows get a gradient?** Four distinct IDs appear in the batch (`0, 1, 5, 7`). Predict which rows of `E.weight.grad` are non-zero after `loss.backward()`.

In [ ]:
w_out_random = torch.randn(V, D, requires_grad=True)
loss = F.cross_entropy((embedding(token_ids) @ w_out_random.T).view(B * T, V), targets.view(B * T))
embedding.zero_grad()
loss.backward()

row_norms = embedding.weight.grad.norm(dim=1)
e03_rows_with_gradient = [i for i, n in enumerate(row_norms.tolist()) if n > 0]
print("rows of E with a non-zero gradient:", e03_rows_with_gradient)
print("token IDs in the batch            :", sorted(set(token_ids.flatten().tolist())))

Only the rows that were looked up receive a gradient. A token that never appears in the training data keeps its random initial vector: one reason why rare tokens have poor embeddings, and why the tokenizer of Lecture 01 matters for the model of Lecture 03.

## E04 · Count the parameters

The smallest possible neural language model has two tables: the input embedding $E\in\mathbb{R}^{|V|\times d}$ and the output projection $W_{out}\in\mathbb{R}^{|V|\times d}$. **Weight sharing** (also called weight tying) uses one table for both: $W_{out}=E$.

Predict the parameter counts for $|V|=10$, $d=4$ with and without sharing, then for two released models.

In [ ]:
class TinyLM(torch.nn.Module):
    # Lookup, then project back to the vocabulary: a bigram model with a d-dimensional bottleneck.

    def __init__(self, vocab_size, dim, share_weights=False):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, dim)
        self.output = torch.nn.Linear(dim, vocab_size, bias=False)   # weight has shape (|V|, d)
        torch.nn.init.normal_(self.embedding.weight, std=0.02)       # small values: logits start near zero
        if share_weights:
            self.output.weight = self.embedding.weight

    def forward(self, token_ids):                                     # (B, T) -> (B, T, |V|)
        return self.output(self.embedding(token_ids))


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


e04_untied = count_parameters(TinyLM(V, D))
e04_tied = count_parameters(TinyLM(V, D, share_weights=True))
print(f"|V| = {V}, d = {D}: separate tables = {e04_untied}, shared table = {e04_tied}")
print("logits shape:", tuple(TinyLM(V, D, share_weights=True)(token_ids).shape))

In [ ]:
# Published configurations: GPT-2 small (Radford et al., 2019) and Qwen3-0.6B (Qwen Team, 2025).
# Both share the input and output tables.
MODELS = {
    "GPT-2 small": {"vocab": 50257, "dim": 768, "total": 124_439_808},
    "Qwen3-0.6B": {"vocab": 151_936, "dim": 1024, "total": None},
}

e04_tables = {}
for name, cfg in MODELS.items():
    table = cfg["vocab"] * cfg["dim"]
    e04_tables[name] = table
    share = f"{table / cfg['total']:.1%} of {cfg['total']:,} parameters" if cfg["total"] else "about 0.16B of 0.6B (model card)"
    print(f"{name:12s} |V| x d = {cfg['vocab']:>7,} x {cfg['dim']:>4} = {table:>11,}  ({share})")
    print(f"{'':12s} memory: {table * 4 / 2**20:7.1f} MiB in fp32, {table * 2 / 2**20:7.1f} MiB in bf16; "
          f"a separate output table would add the same again")

**Check:** $10\times4 = 40$ per table, so $80$ without sharing and $40$ with it. GPT-2 small spends $38{,}597{,}376$ of its $124{,}439{,}808$ parameters (31%) on one shared table. Small models with large vocabularies are dominated by their embedding tables, which is why they usually share weights; very large models often do not, because the table is a small fraction of the total.

Memory is the parameter count times the bytes per number: 4 bytes in fp32, 2 bytes in bf16. Training needs more (gradients and optimizer state), which returns in the weeks on pretraining and compute budgets.

## P01 · Train skip-gram with negative sampling on a toy corpus

This is the slide deck's word2vec recipe in PyTorch: two tables $W$ (center words) and $C$ (context words), positive pairs from a window of $m=3$ (the slides use $m=2$; the toy sentences need one more word of context), $k=5$ negatives per pair sampled from the unigram distribution raised to the power $0.75$, and the loss of E02 averaged over a minibatch.

The corpus is **generated from templates** so that four groups of words (people, animals, foods, places) occur in distinguishable contexts. It is a toy: real embeddings need millions of tokens.

In [ ]:
import random


def build_toy_corpus(sentences_per_template=150, seed=2026):
    rng = random.Random(seed)
    people = {
        "king": ("he", "rules", "palace"), "queen": ("she", "rules", "palace"),
        "prince": ("he", "rules", "palace"), "princess": ("she", "rules", "palace"),
        "man": ("he", "works", "village"), "woman": ("she", "works", "village"),
        "boy": ("he", "works", "village"), "girl": ("she", "works", "village"),
    }
    animals, foods = ["cat", "dog", "horse", "bird"], ["rice", "bread", "fish", "apple"]
    capitals = {"beijing": "china", "paris": "france", "london": "england", "tokyo": "japan"}
    sentences = []
    for _ in range(sentences_per_template):
        person = rng.choice(list(people))
        pronoun, verb, place = people[person]
        age = "young " if person in ("prince", "princess", "boy", "girl") else ""
        sentences.append(f"the {age}{person} said {pronoun} {verb} in the {place}")
        sentences.append(f"the {age}{person} eats {rng.choice(foods)} in the {place}")
        sentences.append(f"the {rng.choice(animals)} chases the {rng.choice(animals)} and eats {rng.choice(foods)}")
        capital = rng.choice(list(capitals))
        sentences.append(f"{capital} is the capital city of {capitals[capital]}")
        sentences.append(f"the {person} travels from the {place} to {rng.choice(list(capitals))}")
    return [s.split() for s in sentences]


TOY_SENTENCES = build_toy_corpus()
VOCAB = sorted({word for sentence in TOY_SENTENCES for word in sentence})
WORD_TO_ID = {word: i for i, word in enumerate(VOCAB)}
print(len(TOY_SENTENCES), "sentences,", sum(map(len, TOY_SENTENCES)), "tokens,", len(VOCAB), "word types")
print(" ".join(TOY_SENTENCES[0]))

In [ ]:
def positive_pairs(sentences, window=3):
    pairs = []
    for sentence in sentences:
        ids = [WORD_TO_ID[word] for word in sentence]
        for i, center in enumerate(ids):
            for j in range(max(0, i - window), min(len(ids), i + window + 1)):
                if j != i:
                    pairs.append((center, ids[j]))
    return torch.tensor(pairs)


PAIRS = positive_pairs(TOY_SENTENCES)
counts = torch.bincount(torch.tensor([WORD_TO_ID[w] for s in TOY_SENTENCES for w in s]), minlength=len(VOCAB)).float()
NOISE = counts.pow(0.75) / counts.pow(0.75).sum()       # P(w) proportional to count^0.75
print("positive pairs:", tuple(PAIRS.shape), "| first pairs:", [(VOCAB[a], VOCAB[b]) for a, b in PAIRS[:4].tolist()])


class SkipGram(torch.nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.W = torch.nn.Embedding(vocab_size, dim)     # center-word vectors
        self.C = torch.nn.Embedding(vocab_size, dim)     # context-word vectors
        torch.nn.init.uniform_(self.W.weight, -0.5 / dim, 0.5 / dim)   # word2vec's initialization:
        torch.nn.init.zeros_(self.C.weight)                            # small W, zero C

    def forward(self, center, positive, negatives):      # (N,), (N,), (N, k)
        w = self.W(center)                               # (N, d)
        positive_score = (w * self.C(positive)).sum(-1)                       # (N,)
        negative_score = torch.einsum("nd,nkd->nk", w, self.C(negatives))     # (N, k)
        return -(F.logsigmoid(positive_score) + F.logsigmoid(-negative_score).sum(-1)).mean()


def train_skipgram(dim=16, k=5, epochs=15, batch_size=256, lr=0.01, seed=0):
    generator = torch.Generator().manual_seed(seed)
    torch.manual_seed(seed)
    model = SkipGram(len(VOCAB), dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(epochs):
        order = torch.randperm(len(PAIRS), generator=generator)
        total = 0.0
        for start in range(0, len(PAIRS), batch_size):
            batch = PAIRS[order[start:start + batch_size]]
            negatives = torch.multinomial(NOISE, len(batch) * k, replacement=True, generator=generator).view(len(batch), k)
            loss = model(batch[:, 0], batch[:, 1], negatives)
            if epoch == 0 and start == 0:
                print(f"first minibatch: loss = {loss.item():.4f}")
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(batch)
        history.append(total / len(PAIRS))
        print(f"epoch {epoch + 1}: mean loss = {history[-1]:.4f}")
    return model, history


skipgram, p01_history = train_skipgram()

With $k=5$ negatives and the context table initialized to zero, every score is zero and the first minibatch has loss $6\log 2 = 4.1589$. The loss falls as positive pairs get higher scores than sampled pairs; it cannot reach zero, because frequent words are also drawn as negatives.

## P02 · Nearest neighbours and analogies on the toy vectors

Cosine similarity compares directions. The analogy query follows the slide: $\mathbf{q} = \mathbf{v}(a^*) - \mathbf{v}(a) + \mathbf{v}(b)$, and the answer is the nearest word that is not one of the three query words.

In [ ]:
VECTORS = F.normalize(skipgram.W.weight.detach(), dim=1)        # unit length, so a dot product is a cosine


def nearest(word, k=3):
    scores = VECTORS @ VECTORS[WORD_TO_ID[word]]
    scores[WORD_TO_ID[word]] = -1.0
    return [(VOCAB[i], round(scores[i].item(), 2)) for i in scores.topk(k).indices.tolist()]


def analogy(a, a_star, b, k=3):
    # a is to a_star as b is to ?
    query = F.normalize(VECTORS[WORD_TO_ID[a_star]] - VECTORS[WORD_TO_ID[a]] + VECTORS[WORD_TO_ID[b]], dim=0)
    scores = VECTORS @ query
    for word in (a, a_star, b):
        scores[WORD_TO_ID[word]] = -1.0
    return [VOCAB[i] for i in scores.topk(k).indices.tolist()]


for word in ["king", "cat", "rice", "paris"]:
    print(f"{word:6s} ->", nearest(word))
print("man : woman = king : ?   ", analogy("man", "woman", "king"))
print("man : woman = prince : ? ", analogy("man", "woman", "prince"))

The groups are recovered because the templates gave each group its own contexts: the distributional hypothesis in miniature. Analogies on sixteen-dimensional toy vectors are fragile; change `seed` or `dim` in `train_skipgram` and see which results survive. Words that share every context (such as `king` and `prince` here) cannot be told apart by any method that only looks at contexts.

## P03 · The five-line training loop on one batch

Every model in this course is trained with the same five lines: clear the gradients, forward pass, loss, backward pass, optimizer step. Here `TinyLM` from E04 memorizes one batch. The loss starts near $\log|V|$ and falls toward zero.

In [ ]:
torch.manual_seed(0)
model = TinyLM(V, D, share_weights=True)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
inputs = torch.tensor([[1, 2, 3], [4, 5, 6]])       # each token is followed by the next integer
labels = torch.tensor([[2, 3, 4], [5, 6, 7]])

p03_losses = []
for step in range(200):
    optimizer.zero_grad()
    logits = model(inputs)                                          # (B, T, |V|)
    loss = F.cross_entropy(logits.view(-1, V), labels.view(-1))
    loss.backward()
    optimizer.step()
    p03_losses.append(loss.item())
    if step in (0, 9, 49, 199):
        print(f"step {step + 1:3d}: loss = {loss.item():.4f}")

print("prediction after 1, 2, 3:", model(inputs)[0].argmax(-1).tolist(), "| labels:", labels[0].tolist())

Overfitting one tiny batch is the first test of any training loop: if the loss does not approach zero here, there is a bug in the model, the loss, or the data pipeline. Week 4 builds a feedforward language model on top of this loop.